In [ ]:
%%sql -r dataframe_1
select * from travel_booking.public.travel_bookings limit 2;

In [ ]:
%%sql -r dataframe_3
select count(*) as total_books from travel_booking.public.travel_bookings

In [ ]:
%%sql -r dataframe_4
select booking_id, customer_name, destination_city, travel_start_date from travel_booking.public.travel_bookings where destination_country = 'Spain';

In [ ]:
%%sql -r dataframe_5
select booking_id, status from travel_booking.public.travel_bookings where status = 'Cancelled';


In [ ]:
%%sql -r dataframe_6
select booking_id, discount_pct from travel_booking.public.travel_bookings where discount_pct IS NULL;


In [ ]:
%%sql -r dataframe_7
select booking_id, travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100) as revenue_gbp from travel_booking.public.travel_bookings;


In [ ]:
%%sql -r dataframe_8
select package_type, ROUND(SUM(travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100)), 2) AS revenue_gbp from travel_booking.public.travel_bookings
where status <> 'Cancelled' group by package_type order by revenue_gbp desc;


In [ ]:
%%sql -r dataframe_9
select destination_country, COUNT(*) as bookings from travel_booking.public.travel_bookings
group by destination_country order by bookings desc
limit 1;


In [ ]:
%%sql -r dataframe_10
select payment_mode, COUNT(*) as bookings, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_share from travel_booking.public.travel_bookings
group by payment_mode order by bookings desc;


In [ ]:
%%sql -r dataframe_11
select customer_name, COUNT(*) as bookings from travel_booking.public.travel_bookings
group by customer_name having COUNT(*) > 2;


In [ ]:
select destination_country, round(avg(rating), 2) as avg_rating, count(rating) as rated_trips from travel_booking.public.travel_bookings
group by destination_country having count(rating) >= 2 order by avg_rating desc;

In [ ]:
select booking_id, customer_name, dateadd('day', nights, travel_start_date) as return_date, datediff('day', booking_date, travel_start_date) as lead_time_days
from travel_booking.public.travel_bookings;

In [ ]:
select date_trunc('month', travel_start_date) as travel_month, count(*) as bookings, round(sum(travellers * price_per_person_gbp * (1 - coalesce(discount_pct, 0) / 100)), 2) as revenue_gbp from travel_booking.public.travel_bookings where status <> 'cancelled'
group by travel_month order by travel_month;

In [ ]:
select home_city, customer_name, total_spend, rank() over (partition by home_city order by total_spend desc) as city_rank from (
select home_city, customer_name, sum(travellers * price_per_person_gbp * (1 - coalesce(discount_pct, 0) / 100)) as total_spend from travel_booking.public.travel_bookings where status <> 'cancelled'
group by home_city, customer_name
);

In [ ]:
%%sql -r dataframe_15
select customer_name, booking_date, datediff('day', lag(booking_date) over (partition by customer_id order by booking_date), booking_date) as days_since_last_booking from travel_booking.public.travel_bookings qualify count(*) over (partition by customer_id) > 1
order by customer_name, booking_date;